In [2]:
import requests
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from dotenv import load_dotenv, find_dotenv 
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

from download_cards import download_model_cards

from retriever import Retriever
from generator import Generator

_ = load_dotenv(find_dotenv())

In [2]:
download_model_cards()

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

# Indexing 

In [3]:

# Step 1.1: load documents
loader = DirectoryLoader('model_cards/', glob="**/*.md", loader_cls=TextLoader)
documents = loader.load()
# Step 1.2: split documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1051, which is longer than the specified 500
Created a chunk of size 650, which is longer than the specified 500
Created a chunk of size 1011, which is longer than the specified 500
Created a chunk of size 824, which is longer than the specified 500
Created a chunk of size 1711, which is longer than the specified 500
Created a chunk of size 1604, which is longer than the specified 500
Created a chunk of size 1587, which is longer than the specified 500
Created a chunk of size 1011, which is longer than the specified 500
Created a chunk of size 824, which is longer than the specified 500
Created a chunk of size 1711, which is longer than the specified 500
Created a chunk of size 1604, which is longer than the specified 500
Created a chunk of size 1587, which is longer than the specified 500
Created a chunk of size 504, which is longer than the specified 500
Created a chunk of size 533, which is longer than the specified 500
Created a chunk of size 730, which is l

In [4]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# Step 1.3: encode chunks into vectors and store in a vector database
vectordb = FAISS.from_documents(documents, embeddings)


# Retrieval

In [4]:
# Step 2: Retrieval: retrieve the Top k chunks most relevant to the question based on semantic similarity.
retriever = vectordb.as_retriever()

# Generation

In [5]:
generator = Generator(model="gpt-4o")
template = generator.format_prompt(system_prompt_path="prompts/system_prompt_codegen.txt", user_prompt_path="prompts/user_prompt_codegen.txt")
template

"You are an assistant for code generation tasks. \nUse the following pieces of retrieved context to answer the question. \nIMPORTANT: If you don't know the answer, just say that you don't know.  \n Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"

## Prompt Engineering
- TODO: DSPy

In [6]:
#template = """You are an assistant for question-answering tasks. 
#Use the following pieces of retrieved context to answer the question. 
#If you don't know the answer, just say that you don't know. 
#Use three sentences maximum and keep the answer concise.
#Question: {question} 
#Context: {context} 
#Answer:
#"""

prompt = ChatPromptTemplate.from_template(template)

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for code generation tasks. \nUse the following pieces of retrieved context to answer the question. \nIMPORTANT: If you don't know the answer, just say that you don't know.  \n Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [14]:
# Step 3: Generation: input the original question and the retrieved chunks together into LLM to generate the final answer.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.5)

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | llm
    | StrOutputParser() 
)

query = "Please generate a new python script that detects data drift for tabular data?"
rag_chain.invoke(query)

'To detect data drift for tabular data, you can use the `evidently` library in Python. First, prepare your reference and production datasets and define the schema with numerical and categorical columns. Then, create a `Report` using `DataDriftPreset` and run it on the datasets to evaluate data drift.'

In [10]:
!ls ..

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


gpt  llama4


# RAG Evaluation

In [ ]:
# Import necessary libraries
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Instantiate the models
generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# Create the TestsetGenerator
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# Call the generator
testset = generator.generate_with_langchain_docs(
data_transformed, 
test_size=20, 
distributions={ 
simple: 0.5, 
reasoning: 0.25, 
multi_context: 0.25}
)